# Experiment 3 -- PhotonFlow + Photonic Noise Injection on MNIST**Paper Table I, Exp 3**: Exp 2 + shot noise & thermal crosstalk.**Baseline**: Exp 2 (no noise).  **Metrics**: FID, training curve overlay.**Key question**: *Does noise-aware training help bridge the simulation-to-hardware gap?*## What changes from Exp 2| Component | Exp 2 (no noise) | Exp 3 (this notebook) ||-----------|-----------------|----------------------|| PhotonicNoise | `use_noise=False` | `use_noise=True` || Shot noise | OFF | `sigma_s=0.02` (Shen 2017) || Thermal crosstalk | OFF | `sigma_t=0.01` (Ning 2024) || Injection point | -- | After **each** Monarch layer || Active during | -- | Training only (eval = pass-through) |## Architecture: `PhotonFlowModel` with noise (photonflow/model.py)```x_t --> [Linear 784->256] --> 6x PhotonFlowBlock --> [Linear 256->784] --> v_thetaEach PhotonFlowBlock (paper Fig. 1 middle panel):  Monarch L (M=PLP^TR, Dao 2022)   <-->  MZI mesh array  >>> PhotonicNoise <<<             <-->  Shot noise + thermal crosstalk  Monarch R                         <-->  MZI mesh array  >>> PhotonicNoise <<<             <-->  Shot noise + thermal crosstalk  SaturableAbsorber tanh(0.8x)/0.8  <-->  Graphene waveguide  DivisivePowerNorm x/(||x||+eps)   <-->  Microring resonator  + time_proj(t_emb)                <-->  Electronic preprocessing  Zero-init residual (alpha=0)      <-->  DiT trick (Peebles 2023)```## Noise models (photonflow/noise.py)**Shot noise** (Shen et al. 2017, Methods):- `noise_shot[i] ~ N(0, sigma_s^2)`, sigma_s = 0.02 (additive, iid)- Physical origin: quantum randomness of photon arrivals at detector- Shen 2017 measured sigma_D ~ 0.001; we use 20x for stronger regularization (Ning 2025 rationale)**Thermal crosstalk** (Ning et al. 2024, Section V.C):- `raw ~ N(0, I)`, `corr = conv1d(raw, [0.25, 0.50, 0.25], circular padding)`- `noise_thermal = sigma_t * corr`, sigma_t = 0.01- Physical origin: heat coupling between adjacent phase shifters on MZI chip## Training objective (same CFM loss, Lipman 2023 Eq. 23)```L(theta) = E_{t, x0, x1} [ || v_theta(x_t, t) - (x1 - x0) ||^2 ]x_t = (1-t)*x0 + t*x1     (OT path, sigma_min=0)```Noise is injected **inside** the model forward pass (after each Monarch layer).The CFM loss itself is unchanged -- noise acts as a regularizer.## References- Shen et al. 2017 -- Deep Learning with Coherent Nanophotonic Circuits (Nature Photonics)- Ning et al. 2024 -- Photonic-Electronic Integrated Circuits (J. Lightwave Technol.)- Ning et al. 2025 -- StrC-ONN: noise-aware training for photonic hardware robustness- Lipman et al. 2023 -- Flow Matching for Generative Modeling (ICLR)- Dao et al. 2022 -- Monarch matrices M=PLP^TR (ICML)- Peebles & Xie 2023 -- DiT zero-init residual trick (ICCV)## Success criterionFID within 10% of exp1 baseline (paper abstract).

In [ ]:
# -- 1. Environment detection + dependency install -------------------------
import sys, subprocess, os

# ---- Detect environment ----
# Check Kaggle FIRST -- Kaggle kernels can import the Colab package,
# so checking Colab first would misdetect Kaggle as Colab.
IN_COLAB  = False
IN_KAGGLE = False
IN_LOCAL  = False

if os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    IN_KAGGLE = True
    ENV_NAME  = "Kaggle"
else:
    try:
        import google.colab
        IN_COLAB = True
        ENV_NAME = "Colab"
    except ImportError:
        IN_LOCAL = True
        ENV_NAME = "Local"

print(f"Environment: {ENV_NAME}")

# ---- Install dependencies (Colab + Kaggle) ----
if IN_COLAB or IN_KAGGLE:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torchcfm", "photontorch", "tqdm", "pyyaml", "wandb"
    ])
    # torchonn from our fork (see README Step 4)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/HasinthakaPiyumal/pytorch-onn.git"
    ])

import torch
assert torch.cuda.is_available(), (
    "GPU not found.\n"
    "  Colab : Runtime > Change runtime type > T4 GPU\n"
    "  Kaggle: Settings > Accelerator > GPU T4 x2"
)
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# -- 2. Repo / sys.path setup -----------------------------------------------
REPO_URL    = "https://github.com/HasinthakaPiyumal/photon-flow-research.git"
REPO_BRANCH = "h/phase1"

if IN_COLAB:
    REPO_DIR = "/content/photon-flow-research"
elif IN_KAGGLE:
    REPO_DIR = "/kaggle/working/photon-flow-research"
else:
    REPO_DIR = ".."

if (IN_COLAB or IN_KAGGLE) and not os.path.exists(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR
    ])
    print(f"Cloned repo (branch={REPO_BRANCH}) to {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo  : {os.path.abspath(REPO_DIR)}")
print(f"Branch: {REPO_BRANCH}")

In [ ]:
# -- 3. Imports ---------------------------------------------------------------
import math, json, copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# PhotonFlow core
from photonflow.model import PhotonFlowModel      # Dao 2022 Monarch architecture
from photonflow.train import CFMLoss, euler_sample  # Lipman 2023 CFM training
from photonflow.noise import PhotonicNoise          # Shen 2017 + Ning 2024 noise models

# FID evaluation (Heusel 2017)
from eval.fid import FIDCalculator

device = torch.device("cuda")
print(f"Device: {device}")
print("Imports OK -- PhotonFlowModel + CFMLoss + euler_sample + PhotonicNoise loaded")

In [ ]:
# -- W&B init (get API key from secrets) ------------------------------------
import wandb

# ---- Retrieve WANDB_API_KEY from the environment secrets ----
# Colab : Add-ons > Secrets > "WANDB_API_KEY"
# Kaggle: Add-ons > Secrets > "WANDB_API_KEY" (toggle Internet ON)
# Local : export WANDB_API_KEY=... in shell
WANDB_API_KEY = None

if IN_COLAB:
    try:
        from google.colab import userdata
        WANDB_API_KEY = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print(f"[warn] Could not load WANDB_API_KEY from Colab secrets: {e}")

elif IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception as e:
        print(f"[warn] Could not load WANDB_API_KEY from Kaggle secrets: {e}")

else:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("W&B login: OK")
else:
    print("[warn] WANDB_API_KEY not found -- running wandb in offline mode")
    os.environ["WANDB_MODE"] = "offline"

wandb_run = None
print("W&B cell loaded -- run will be initialised after config is ready")

In [ ]:
# -- 4. Experiment config (from configs/exp3_noise.yaml) + wandb run init ---
import yaml

config_path = os.path.join(REPO_DIR, "configs", "exp3_noise.yaml")
with open(config_path) as f:
    yaml_cfg = yaml.safe_load(f)

print(f"Loaded config: {config_path}")
_exp_name = yaml_cfg["experiment"]["name"]
print(f"  experiment: {_exp_name}")

# Flatten nested YAML into a single CFG dict
CFG = {
    # Model
    "in_dim"     : yaml_cfg["model"]["in_dim"],
    "hidden_dim" : yaml_cfg["model"]["hidden_dim"],
    "num_blocks" : yaml_cfg["model"]["num_blocks"],
    "use_noise"  : yaml_cfg["model"]["use_noise"],
    "sigma_s"    : yaml_cfg["model"]["sigma_s"],
    "sigma_t"    : yaml_cfg["model"]["sigma_t"],
    # Data
    "dataset"    : yaml_cfg["data"]["dataset"],
    "batch_size" : yaml_cfg["data"]["batch_size"],
    "data_root"  : os.path.join(REPO_DIR, yaml_cfg["data"]["root"]),
    # Training
    "lr"               : yaml_cfg["training"]["lr"],
    "total_steps"      : yaml_cfg["training"]["total_steps"],
    "checkpoint_every" : yaml_cfg["training"]["checkpoint_every"],
    "sample_every"     : yaml_cfg["training"]["sample_every"],
    "sample_steps"     : yaml_cfg["training"]["sample_steps"],
    "seed"             : yaml_cfg["training"]["seed"],
}
# time_dim: use from YAML if present, else default to hidden_dim
CFG["time_dim"] = yaml_cfg["model"].get("time_dim", CFG["hidden_dim"])

# ---- Ensure 100K steps (matches YAML config) ----
CFG["total_steps"] = 100_000

IN_DIM = CFG["in_dim"]

# Output directories
OUTPUT_DIR = os.path.join(REPO_DIR, yaml_cfg["output_dir"])
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
for d in [CKPT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

# Print config summary (Python 3.10-safe -- no nested quotes in f-strings)
_hd = CFG["hidden_dim"]; _nb = CFG["num_blocks"]
_un = CFG["use_noise"]; _ss = CFG["sigma_s"]; _st = CFG["sigma_t"]
_ts = CFG["total_steps"]; _lr = CFG["lr"]; _bs = CFG["batch_size"]
print(f"Model: PhotonFlowModel, hidden={_hd}, blocks={_nb}")
print(f"Noise: use_noise={_un}, sigma_s={_ss}, sigma_t={_st}")
print(f"Training: {_ts:,} steps, bs={_bs}, lr={_lr}")
print(f"Output: {OUTPUT_DIR}")

# ---- Init W&B run ----
import wandb
wandb_run = wandb.init(
    project = "photonflow",
    name    = f"{_exp_name}_{ENV_NAME.lower()}",
    config  = {**CFG, "environment": ENV_NAME},
    tags    = ["photonflow", "noise", "mnist", "cfm", ENV_NAME.lower()],
    notes   = yaml_cfg["experiment"]["description"],
    reinit  = True,
)
_wurl = wandb_run.url
print(f"W&B run: {wandb_run.name}  url={_wurl}")

In [ ]:
# -- 5. MNIST DataLoader ----------------------------------------------------
_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda img: img.view(-1)),  # (1,28,28) -> (784,)
])

train_ds = datasets.MNIST(CFG["data_root"], train=True,  download=True, transform=_tfm)
test_ds  = datasets.MNIST(CFG["data_root"], train=False, download=True, transform=_tfm)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"],
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)

steps_per_epoch = len(train_loader)
total_epochs    = CFG["total_steps"] / steps_per_epoch
_bs = CFG["batch_size"]; _ts = CFG["total_steps"]
print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}")
print(f"Batch size: {_bs}  Steps/epoch: {steps_per_epoch}")
print(f"Total epochs: {total_epochs:.1f}  ({_ts:,} steps)")

In [ ]:
# -- 6. PhotonFlowModel instantiation (NOISE ENABLED) ----------------------
# This is the KEY difference from exp2: use_noise=True
# Noise is injected after each Monarch layer during training only.
# At eval time, PhotonicNoise is a no-op (pass-through).

model = PhotonFlowModel(
    in_dim     = CFG["in_dim"],       # 784 (MNIST)
    hidden_dim = CFG["hidden_dim"],   # 256 = 16^2
    num_blocks = CFG["num_blocks"],   # 6
    time_dim   = CFG["time_dim"],     # 256
    use_noise  = CFG["use_noise"],    # True -- KEY difference from exp2
    sigma_s    = CFG["sigma_s"],      # 0.02 shot noise (Shen 2017)
    sigma_t    = CFG["sigma_t"],      # 0.01 thermal crosstalk (Ning 2024)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
loss_fn   = CFMLoss(sigma_min=0.0)   # Lipman 2023 Eq. 23 OT-CFM

total_params = model.count_parameters()
m = math.isqrt(CFG["hidden_dim"])
monarch_params = 2 * m**3   # per MonarchLayer (L + R, no bias)
dense_params   = CFG["hidden_dim"]**2

_hd = CFG["hidden_dim"]; _nb = CFG["num_blocks"]
_un = CFG["use_noise"]; _ss = CFG["sigma_s"]; _st = CFG["sigma_t"]
print(f"Model: PhotonFlowModel")
print(f"  hidden_dim = {_hd} = {m}^2")
print(f"  blocks     = {_nb}")
print(f"  noise      = {_un} (sigma_s={_ss}, sigma_t={_st})")
print(f"  params     = {total_params:,}")
print(f"  Monarch params/layer: {monarch_params:,} vs dense {dense_params:,} = {dense_params/monarch_params:.1f}x fewer")

# Verify noise modules exist in each block
for i, blk in enumerate(model.blocks):
    assert blk.noise_l is not None, f"Block {i}: noise_l is None"
    assert blk.noise_r is not None, f"Block {i}: noise_r is None"
    _nls = blk.noise_l.sigma_s; _nlt = blk.noise_l.sigma_t
    print(f"  Block {i}: noise_l(sigma_s={_nls}, sigma_t={_nlt}), noise_r OK")

# Zero-init verification (eval mode for deterministic check)
with torch.no_grad():
    _x = torch.randn(4, IN_DIM, device=device)
    _t = torch.rand(4, device=device)
    model.eval()
    _out = model(_x, _t)
    _max = _out.abs().max().item()
    model.train()
print(f"\nInit output max abs: {_max:.6f}  (should be ~0 -- zero-init)")
assert _max < 1e-3, f"Expected near-zero init, got {_max}"
print("Initialization OK -- noise modules present + zero-init confirmed")

In [ ]:
# -- 7. VERIFY NOISE INJECTION ---------------------------------------------
# Prove that PhotonicNoise is active in train mode and inactive in eval mode.
# This is the core validation that exp3 differs from exp2.
#
# Shot noise:      noise_i ~ N(0, sigma_s^2), sigma_s=0.02  (Shen 2017)
# Thermal crosstalk: correlated via kernel [0.25, 0.50, 0.25], sigma_t=0.01 (Ning 2024)

print("Noise injection verification")
print("=" * 50)

_test_x = torch.randn(8, IN_DIM, device=device)
_test_t = torch.rand(8, device=device)

# --- Test A: eval mode -> deterministic (identical outputs) ---
model.eval()
with torch.no_grad():
    _out_a = model(_test_x, _test_t)
    _out_b = model(_test_x, _test_t)
assert torch.equal(_out_a, _out_b), "FAIL: eval mode not deterministic"
print("[PASS] Test A -- Eval mode: deterministic (two runs identical)")

# --- Test B: train mode -> stochastic (different outputs due to noise) ---
# Use a temporary deep copy to avoid disturbing the real model state.
# Must set alpha > 0 and output_proj non-zero to observe noise propagation
# (alpha=0 zeros out the block path; zero output_proj zeros output).
_test_model = copy.deepcopy(model)
_test_model.train()
for blk in _test_model.blocks:
    blk.alpha.data.fill_(1.0)
nn.init.normal_(_test_model.output_proj.weight)
nn.init.ones_(_test_model.output_proj.bias)

with torch.no_grad():
    _out_c = _test_model(_test_x, _test_t)
    _out_d = _test_model(_test_x, _test_t)
assert not torch.equal(_out_c, _out_d), "FAIL: train mode not stochastic"
_noise_mag = (_out_c - _out_d).abs().mean().item()
print(f"[PASS] Test B -- Train mode: stochastic (mean diff = {_noise_mag:.6f})")

# --- Test C: empirical noise magnitude from isolated PhotonicNoise ---
_ss = CFG["sigma_s"]; _st = CFG["sigma_t"]
_noise_fn = PhotonicNoise(sigma_s=_ss, sigma_t=_st)
_noise_fn.train()
_zero_input = torch.zeros(10000, CFG["hidden_dim"])
_noisy = _noise_fn(_zero_input) - _zero_input
_empirical_std = _noisy.std().item()
# Expected: sqrt(sigma_s^2 + 0.375*sigma_t^2)
# kernel [0.25,0.50,0.25] has L2-norm^2 = 0.0625+0.25+0.0625 = 0.375
_expected_std = math.sqrt(_ss**2 + 0.375 * _st**2)
print(f"[PASS] Test C -- Empirical combined noise std: {_empirical_std:.4f}")
print(f"       Expected: sqrt(sigma_s^2 + 0.375*sigma_t^2) = {_expected_std:.4f}")

# --- Test D: thermal noise is correlated (adjacent elements co-vary) ---
_thermal_only = PhotonicNoise(sigma_s=0.0, sigma_t=_st)
_thermal_only.train()
_tn = _thermal_only(torch.zeros(50000, 32))
_col_i = _tn[:, 15] - _tn[:, 15].mean()
_col_j = _tn[:, 16] - _tn[:, 16].mean()
_corr = (_col_i * _col_j).mean() / (_col_i.std() * _col_j.std() + 1e-8)
print(f"[PASS] Test D -- Thermal correlation: corr(noise[15], noise[16]) = {_corr.item():.3f} (expected ~0.667)")

del _test_model   # free GPU memory
model.train()     # ensure model is back in train mode

print()
print("=" * 50)
print("All noise checks passed -- noise injection is active and correct")
print("=" * 50)

In [ ]:
# -- 8. VERIFY FIRST 1K STEPS (quick sanity check) -------------------------
# Run 1000 steps with noise ON before committing to the full 100K run.
# Check: loss decreases, no NaN/Inf, gradients flow, samples not garbage.

VERIFY_STEPS = 1000

verify_losses = []
data_iter_v = iter(train_loader)
model.train()

print(f"Verification run: {VERIFY_STEPS} steps (with noise ON) ...")
print()

for step in range(VERIFY_STEPS):
    try:
        x1, _ = next(data_iter_v)
    except StopIteration:
        data_iter_v = iter(train_loader)
        x1, _ = next(data_iter_v)

    x1 = x1.to(device)
    loss = loss_fn(model, x1)

    optimizer.zero_grad()
    loss.backward()

    # Check gradients on first step
    if step == 0:
        no_grad = [n for n, p in model.named_parameters() if p.grad is None]
        assert len(no_grad) == 0, f"Params with no gradient: {no_grad}"
        for n, p in model.named_parameters():
            assert not torch.isnan(p.grad).any(), f"NaN grad in {n}"
            assert not torch.isinf(p.grad).any(), f"Inf grad in {n}"

    optimizer.step()
    verify_losses.append(loss.item())
    wandb.log({"verify/loss": loss.item()}, step=step)

    if (step + 1) % 200 == 0:
        avg = np.mean(verify_losses[-200:])
        print(f"  step {step+1:>5d}/{VERIFY_STEPS}  loss={avg:.4f}")

# --- Verdict ---
first_100 = np.mean(verify_losses[:100])
last_100  = np.mean(verify_losses[-100:])
any_nan   = any(math.isnan(l) for l in verify_losses)
decreased = last_100 < first_100

print()
print(f"  Loss first 100: {first_100:.4f}")
print(f"  Loss last  100: {last_100:.4f}")
print(f"  Decreased:      {decreased}")
print(f"  Any NaN:        {any_nan}")
print(f"  Gradients:      OK (checked on step 0)")

# Quick sample check
model.eval()
with torch.no_grad():
    quick_samp = euler_sample(
        model, shape=(16, IN_DIM),
        num_steps=CFG["sample_steps"], device=str(device)
    )
quick_samp = quick_samp.clamp(0.0, 1.0).cpu().view(16, 1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(quick_samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle(f"Exp3 verification (1K steps, noise ON, loss={last_100:.4f})", fontsize=11)
plt.tight_layout()
wandb.log({"verify/samples": wandb.Image(fig, caption="1K-step verification")})
plt.show()
plt.close(fig)
model.train()

# GO / NO-GO
if decreased and not any_nan:
    print()
    print("=" * 50)
    print("  VERDICT: GO -- proceed to full training (Cell 9)")
    print("=" * 50)
else:
    print()
    print("=" * 50)
    print("  VERDICT: NO-GO -- debug before full training")
    print("=" * 50)
    if any_nan:
        print("  Problem: NaN in loss -- try lower lr (1e-5)")
    if not decreased:
        print("  Problem: loss not decreasing -- check model/data")

# Reset model for full training (start from scratch)
model = PhotonFlowModel(
    in_dim=CFG["in_dim"], hidden_dim=CFG["hidden_dim"],
    num_blocks=CFG["num_blocks"], time_dim=CFG["time_dim"],
    use_noise=CFG["use_noise"], sigma_s=CFG["sigma_s"], sigma_t=CFG["sigma_t"],
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
torch.manual_seed(CFG["seed"])
print("\nModel reset for full training in Cell 9.")

In [ ]:
# -- 9. Training loop (100K steps, noise ON) --------------------------------
# Same CFM loss as exp1/exp2 (Lipman 2023 Eq. 23) -- architecture-agnostic.
# The ONLY difference from exp2: PhotonicNoise is active inside each block.
# Noise acts as a regularizer -- the model learns weight configurations
# that are robust to shot noise + thermal crosstalk on real photonic hardware.

import wandb
losses   = []
step_log = []

data_iter = iter(train_loader)
model.train()
pbar = tqdm(range(CFG["total_steps"]), desc="exp3 PhotonFlow+noise", dynamic_ncols=True)

for step in pbar:
    try:
        x1, _ = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        x1, _ = next(data_iter)

    x1 = x1.to(device)  # (B, 784)

    # CFM loss (Lipman 2023 Eq. 23) -- noise injected inside model forward
    loss = loss_fn(model, x1)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())

    # Log every step
    wandb.log({"train/loss": loss.item()}, step=step)

    # Avg every 100 steps
    if step % 100 == 0:
        avg = float(np.mean(losses[max(0, len(losses)-100):]))
        step_log.append((step, avg))
        pbar.set_postfix(loss=f"{avg:.4f}")
        wandb.log({"train/loss_avg100": avg}, step=step)

    # Checkpoint every 5K steps
    if (step + 1) % CFG["checkpoint_every"] == 0:
        ckpt = {
            "step"                : step + 1,
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "losses"              : losses,
            "config"              : CFG,
        }
        ckpt_path = os.path.join(CKPT_DIR, f"step_{step+1:07d}.pt")
        torch.save(ckpt, ckpt_path)
        art = wandb.Artifact(f"exp3-checkpoint-step{step+1}", type="model")
        art.add_file(ckpt_path)
        wandb.log_artifact(art)
        print(f"\n[step {step+1:,}] Checkpoint saved")

    # Sample grid every 5K steps
    if (step + 1) % CFG["sample_every"] == 0:
        model.eval()
        with torch.no_grad():
            samp = euler_sample(
                model, shape=(64, IN_DIM),
                num_steps=CFG["sample_steps"], device=str(device)
            )
        samp = samp.clamp(0.0, 1.0).cpu().view(64, 1, 28, 28)
        fig, axes = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(axes.flat):
            ax.imshow(samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
        avg_now = float(np.mean(losses[max(0, len(losses)-500):]))
        fig.suptitle(f"Exp3 step={step+1:,}  loss={avg_now:.4f}  (noise ON)", fontsize=11)
        plt.tight_layout()
        fig_path = os.path.join(FIG_DIR, f"samples_step_{step+1:07d}.png")
        plt.savefig(fig_path, dpi=100, bbox_inches="tight")
        wandb.log({"samples/grid": wandb.Image(fig, caption=f"step={step+1}")}, step=step)
        plt.show()
        plt.close(fig)
        model.train()

pbar.close()
final_avg = float(np.mean(losses[-1000:]))
print(f"\nTraining complete. Final avg loss (last 1K steps): {final_avg:.4f}")
wandb.log({"train/final_loss_avg1000": final_avg})

In [ ]:
# -- 10. Loss curve ----------------------------------------------------------
import wandb
steps_arr  = [s for s, _ in step_log]
losses_arr = [l for _, l in step_log]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(steps_arr, losses_arr, lw=1.5, alpha=0.85, color="crimson")
ax.set_xlabel("Training step", fontsize=12)
ax.set_ylabel("CFM loss (MSE)", fontsize=12)
_ts_k = CFG["total_steps"] // 1000
ax.set_title(f"Exp3: PhotonFlow + Noise Injection on MNIST ({_ts_k}K steps)", fontsize=13)
ax.grid(True, alpha=0.3)
_final_avg = np.mean(losses[-1000:])
ax.annotate(
    f"final avg = {_final_avg:.4f}",
    xy=(steps_arr[-1], losses_arr[-1]),
    xytext=(-120, 10), textcoords="offset points", fontsize=10, color="crimson")
plt.tight_layout()
curve_path = os.path.join(FIG_DIR, "loss_curve.png")
plt.savefig(curve_path, dpi=150)
wandb.log({"charts/loss_curve": wandb.Image(fig, caption="Exp3 training loss curve")})
plt.show()
print(f"Loss curve saved: {curve_path}")

In [ ]:
# -- 11. Training curve comparison: exp2 vs exp3 ----------------------------
# Load exp2 losses if available and overlay training curves.
# This is Paper deliverable #33: outputs/figures/train_curve_exp2_vs_exp3.png

import wandb

exp2_losses_path = os.path.join(REPO_DIR, "outputs", "exp2_photonflow_mnist", "losses_exp2.npy")

fig, ax = plt.subplots(figsize=(11, 4))

# Exp3 (this experiment) -- crimson
ax.plot(steps_arr, losses_arr, lw=1.5, alpha=0.85, color="crimson",
        label="Exp3: PhotonFlow + noise")

if os.path.exists(exp2_losses_path):
    exp2_losses_raw = np.load(exp2_losses_path)
    # Downsample to avg-per-100 to match step_log format
    exp2_steps = list(range(0, len(exp2_losses_raw), 100))
    exp2_avgs  = [float(np.mean(exp2_losses_raw[max(0, s-100):s+1]))
                  for s in exp2_steps]
    ax.plot(exp2_steps, exp2_avgs, lw=1.5, alpha=0.85, color="darkorange",
            label="Exp2: PhotonFlow (no noise)")
    print(f"Loaded exp2 losses: {len(exp2_losses_raw):,} steps")
    # Annotate final values
    _exp2_final = exp2_avgs[-1]
    _exp3_final = losses_arr[-1]
    _delta = _exp3_final - _exp2_final
    _delta_pct = (_delta / _exp2_final) * 100 if _exp2_final > 0 else 0
    print(f"Exp2 final avg: {_exp2_final:.4f}")
    print(f"Exp3 final avg: {_exp3_final:.4f}")
    print(f"Delta: {_delta:+.4f} ({_delta_pct:+.1f}%)")
else:
    print(f"(exp2 losses not found at {exp2_losses_path} -- run notebook 03 first)")

ax.set_xlabel("Training step", fontsize=12)
ax.set_ylabel("CFM loss (MSE)", fontsize=12)
ax.set_title("Training Curves: Exp2 (no noise) vs Exp3 (noise injection)", fontsize=13)
ax.legend(fontsize=10, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
compare_path = os.path.join(FIG_DIR, "loss_comparison_exp2_vs_exp3.png")
plt.savefig(compare_path, dpi=150)
wandb.log({"charts/loss_comparison": wandb.Image(fig, caption="Exp2 vs Exp3 training curves")})
plt.show()
print(f"Comparison plot saved: {compare_path}")

In [ ]:
# -- 12. Final sample grid (10x10) ------------------------------------------
import wandb
model.eval()
with torch.no_grad():
    final_samp = euler_sample(
        model, shape=(100, IN_DIM),
        num_steps=CFG["sample_steps"], device=str(device)
    )
final_samp = final_samp.clamp(0.0, 1.0).cpu().view(100, 1, 28, 28)

fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(final_samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
_ts_k = CFG["total_steps"] // 1000
fig.suptitle(f"Exp3 Final Samples -- PhotonFlow + Noise ({_ts_k}K steps)", fontsize=13)
plt.tight_layout()
samp_path = os.path.join(FIG_DIR, "final_samples.png")
plt.savefig(samp_path, dpi=150, bbox_inches="tight")
wandb.log({"samples/final_10x10": wandb.Image(fig, caption="Final 100 generated samples")})
plt.show()
print(f"Final samples saved: {samp_path}")

In [ ]:
# -- 13. FID computation ----------------------------------------------------
# FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*(Sigma_r Sigma_g)^(1/2))
# InceptionV3 pool3 features (2048-dim)  |  Heusel et al. 2017
import wandb

N_FID    = 10_000
fid_calc = FIDCalculator(device=device)

# Real images (MNIST test set)
real_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)
real_imgs = []
for imgs, _ in real_loader:
    real_imgs.append(imgs.view(-1, 1, 28, 28))
real_imgs = torch.cat(real_imgs, dim=0)[:N_FID]
print(f"Real images: {real_imgs.shape}")

# Generated images from PhotonFlow (noise OFF during generation -- eval mode)
model.eval()
gen_batches = []
GEN_BATCH   = 256
n_generated = 0
with torch.no_grad():
    pbar_fid = tqdm(total=N_FID, desc="Generating for FID", unit="img")
    while n_generated < N_FID:
        bs   = min(GEN_BATCH, N_FID - n_generated)
        samp = euler_sample(
            model, shape=(bs, IN_DIM),
            num_steps=CFG["sample_steps"], device=str(device)
        )
        gen_batches.append(samp.clamp(0.0, 1.0).cpu().view(bs, 1, 28, 28))
        n_generated += bs
        pbar_fid.update(bs)
    pbar_fid.close()

gen_imgs = torch.cat(gen_batches, dim=0)[:N_FID]
print(f"Generated images: {gen_imgs.shape}")

# Extract features + compute FID
print("Extracting real features...")
real_feats = fid_calc.extract_features(real_imgs, batch_size=256)
print("Extracting generated features...")
gen_feats  = fid_calc.extract_features(gen_imgs, batch_size=256)

real_stats = fid_calc.compute_statistics(real_feats)
gen_stats  = fid_calc.compute_statistics(gen_feats)
fid_score  = fid_calc.compute_fid(real_stats, gen_stats)

_ts_k = CFG["total_steps"] // 1000
print(f"\n*** Exp3 PhotonFlow+Noise FID: {fid_score:.2f} ***")

# Compare vs exp1 baseline
exp1_results_path = os.path.join(REPO_DIR, "outputs", "exp1_baseline", "results_exp1.json")
if os.path.exists(exp1_results_path):
    with open(exp1_results_path) as f:
        exp1_results = json.load(f)
    exp1_fid = exp1_results["fid"]
    delta_pct = ((fid_score - exp1_fid) / exp1_fid) * 100
    target = exp1_fid * 1.10
    passed = fid_score <= target
    status = "PASS" if passed else "FAIL"
    print(f"\nExp1 baseline FID: {exp1_fid:.2f}")
    print(f"  Exp3 delta vs exp1: {delta_pct:+.1f}%  (target: within +10%)")
    print(f"  10% target: <= {target:.2f}  [{status}]")
else:
    print("\n(exp1 results not found -- run notebook 02 first for comparison)")

# Compare vs exp2 (no noise)
exp2_results_path = os.path.join(REPO_DIR, "outputs", "exp2_photonflow_mnist", "results_exp2.json")
if os.path.exists(exp2_results_path):
    with open(exp2_results_path) as f:
        exp2_results = json.load(f)
    exp2_fid = exp2_results["fid"]
    delta_exp2 = ((fid_score - exp2_fid) / exp2_fid) * 100
    print(f"\nExp2 (no noise) FID: {exp2_fid:.2f}")
    print(f"  Exp3 delta vs exp2: {delta_exp2:+.1f}%")
    if fid_score < exp2_fid:
        print("  Noise injection IMPROVED FID (lower is better)")
    elif fid_score > exp2_fid * 1.05:
        print("  Noise injection increased FID by >5% -- expected trade-off for hardware robustness")
    else:
        print("  Noise injection had minimal impact on FID (<5%)")
else:
    print("\n(exp2 results not found -- run notebook 03 first for comparison)")

wandb.log({
    "eval/fid"           : fid_score,
    "eval/n_fid_samples" : N_FID,
})

In [ ]:
# -- 14. Results summary + W&B finish ---------------------------------------
import json as _json, wandb

results = {
    "experiment"   : "exp3_noise_regularized",
    "environment"  : ENV_NAME,
    "description"  : "PhotonFlow + photonic noise injection (shot + thermal crosstalk)",
    "total_steps"  : CFG["total_steps"],
    "fid"          : round(float(fid_score), 4),
    "final_loss_avg1000": round(float(np.mean(losses[-1000:])), 6),
    "model_params" : total_params,
    "architecture" : {
        "type"       : "photonflow",
        "hidden_dim" : CFG["hidden_dim"],
        "num_blocks" : CFG["num_blocks"],
        "time_dim"   : CFG["time_dim"],
        "use_noise"  : CFG["use_noise"],
        "sigma_s"    : CFG["sigma_s"],
        "sigma_t"    : CFG["sigma_t"],
        "monarch_block_size" : math.isqrt(CFG["hidden_dim"]),
    },
    "training" : {
        "optimizer"  : "Adam",
        "lr"         : CFG["lr"],
        "batch_size" : CFG["batch_size"],
    },
    "noise" : {
        "shot_noise_type"       : "additive_iid_gaussian",
        "shot_noise_sigma"      : CFG["sigma_s"],
        "thermal_crosstalk_type": "correlated_gaussian_conv1d",
        "thermal_kernel"        : [0.25, 0.50, 0.25],
        "thermal_sigma"         : CFG["sigma_t"],
        "injection_points"      : "after_each_monarch_layer",
        "active_during"         : "training_only",
    },
    "sources" : [
        "Lipman et al. 2023 -- Flow Matching CFM loss (ICLR, Eq. 23)",
        "Dao et al. 2022 -- Monarch matrices M=PLP^TR (ICML, Def 3.1)",
        "Shen et al. 2017 -- MZI hardware + shot noise (Nature Photonics)",
        "Ning et al. 2024 -- Gaussian noise model for photonics (JLT)",
        "Ning et al. 2025 -- Noise-aware training StrC-ONN",
        "Peebles & Xie 2023 -- Zero-init residual trick (ICCV, DiT)",
    ],
}

results_path = os.path.join(OUTPUT_DIR, "results_exp3.json")
with open(results_path, "w") as f:
    _json.dump(results, f, indent=2)

np.save(os.path.join(OUTPUT_DIR, "losses_exp3.npy"), np.array(losses))

final_ckpt = os.path.join(CKPT_DIR, "exp3_final.pt")
torch.save({
    "step": CFG["total_steps"],
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fid": fid_score,
    "config": CFG,
}, final_ckpt)

# W&B summary
_ss = CFG["sigma_s"]; _st = CFG["sigma_t"]
wandb.log({
    "summary/fid"          : fid_score,
    "summary/final_loss"   : np.mean(losses[-1000:]),
    "summary/model_params" : total_params,
    "summary/total_steps"  : CFG["total_steps"],
    "summary/sigma_s"      : _ss,
    "summary/sigma_t"      : _st,
})
wandb.save(results_path)

SEP = "=" * 58
print(SEP)
print("EXP3 NOISE-REGULARIZED PHOTONFLOW RESULTS")
print(SEP)
_fid = fid_score; _fl = np.mean(losses[-1000:])
_nb = CFG["num_blocks"]; _m = math.isqrt(CFG["hidden_dim"])
_ts = CFG["total_steps"]
print(f"  Environment:           {ENV_NAME}")
print(f"  FID (10K samples):     {_fid:.2f}")
print(f"  Final CFM loss (1K):   {_fl:.4f}")
print(f"  Model parameters:      {total_params:,}")
print(f"  Training steps:        {_ts:,}")
print(f"  Architecture:          PhotonFlow ({_nb} blocks, Monarch m={_m})")
print(f"  Shot noise:            sigma_s={_ss}")
print(f"  Thermal crosstalk:     sigma_t={_st}")
print(SEP)
print(f"  Results    -> {results_path}")
_losses_path = os.path.join(OUTPUT_DIR, "losses_exp3.npy")
print(f"  Losses     -> {_losses_path}")
print(f"  Checkpoint -> {final_ckpt}")
_wurl = wandb_run.url
print(f"  W&B run    -> {_wurl}")
print(SEP)

wandb.finish()

print()
print("Next: notebooks/05_exp4_qat_finetune.ipynb")
print("      Add 4-bit QAT fine-tuning on top of this noise-regularized model")
print("      (load exp3_final.pt checkpoint as starting point)")